In [1]:
%pip install -U google-genai google-cloud-bigquery

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.4/275.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 32.7 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
  Attempting uninstall: google-cloud-bigquery
    Found existing installation: google-cloud-bigquery 3.43.0
    Uninstalling google-cloud-bigquery-3.43.0:
      Successfully uninstalled google-cloud-bigquery-3.43.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This 

CONFIG AND CLIENTS

In [2]:
from google import genai
from google.genai import types
from google.cloud import bigquery

PROJECT_ID = "fintech-lakehouse-dev"
TABLE_NAME = "fintech-lakehouse-dev.fintech_lakehouse_bucket.silver.decoded_transactions"

bq_client = bigquery.Client(project=PROJECT_ID)
print("BigQuery client ready for project:", PROJECT_ID)

BigQuery client ready for project: fintech-lakehouse-dev


TOOL FUNCTION

In [3]:
import datetime as dt
from decimal import Decimal
from typing import Optional
from google.cloud import bigquery
from google.genai import types

UTC = dt.timezone.utc
P = bigquery.ScalarQueryParameter

# Whitelisted group-by expressions: the model picks a key, never writes SQL
GROUP_BY = {
    "merchant_category": "merchant_category",
    "merchant_name":     "merchant_name",
    "card_type":         "card_type",
    "location_country":  "location_country",
    "is_international":  "CAST(is_international AS STRING)",
    "day":   "CAST(DATE(event_time_utc) AS STRING)",
    "week":  "CAST(DATE_TRUNC(DATE(event_time_utc), WEEK(MONDAY)) AS STRING)",
    "month": "FORMAT_DATE('%Y-%m', DATE(event_time_utc))",
    "none":  "'ALL'",
}
TIME_GROUPS = {"day", "week", "month"}


def _clean(v):
    if isinstance(v, Decimal):
        return float(v)
    if isinstance(v, (dt.datetime, dt.date)):
        return v.isoformat()
    return v

def _run(sql, params):
    job = bq_client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params))
    return [{k: _clean(v) for k, v in r.items()} for r in job.result()]

def _window(days, start_date, end_date):
    """Returns (start, end) as UTC datetimes; end is exclusive. Defaults to last 30 days."""
    try:
        if start_date or end_date:
            if start_date:
                start = dt.datetime.fromisoformat(start_date).replace(tzinfo=UTC)
            else:
                start = dt.datetime(2000, 1, 1, tzinfo=UTC)

            if end_date:
                end = dt.datetime.fromisoformat(end_date).replace(tzinfo=UTC) + dt.timedelta(days=1)
            else:
                end = dt.datetime.now(UTC)
        else:
            end = dt.datetime.now(UTC)
            start = end - dt.timedelta(days=int(days or 30))
    except ValueError:
        raise ValueError("Dates must be ISO format YYYY-MM-DD.")
    return start, end

def _filters(merchant_category, merchant_name, card_type, currency, is_international):
    sql, params = "", []
    if merchant_category:
        sql += " AND LOWER(merchant_category) = LOWER(@cat)"
        params.append(P("cat", "STRING", merchant_category))
    if merchant_name:
        sql += " AND LOWER(merchant_name) LIKE @mname"
        params.append(P("mname", "STRING", f"%{merchant_name.lower()}%"))
    if card_type:
        sql += " AND LOWER(card_type) = LOWER(@ctype)"
        params.append(P("ctype", "STRING", card_type))
    if currency:
        sql += " AND UPPER(currency) = UPPER(@cur)"
        params.append(P("cur", "STRING", currency))
    if is_international is not None:
        sql += " AND is_international = @intl"
        params.append(P("intl", "BOOL", bool(is_international)))
    return sql, params


def build_tools(customer_id: str):
    """Creates tool functions bound to ONE customer. The model cannot change customer_id."""

    def get_account_overview() -> dict:
        """Returns the date range of available transactions, total transaction count,
        and the distinct categories, card types and currencies for this customer.
        Call this to learn valid filter values or when the question is about 'overall' activity."""
        sql = f"""
            SELECT MIN(event_time_utc) AS first_txn, MAX(event_time_utc) AS last_txn,
                   COUNT(*) AS total_txn,
                   ARRAY_AGG(DISTINCT merchant_category IGNORE NULLS) AS categories,
                   ARRAY_AGG(DISTINCT card_type IGNORE NULLS) AS card_types,
                   ARRAY_AGG(DISTINCT currency IGNORE NULLS) AS currencies
            FROM `{TABLE_NAME}` WHERE customer_id = @cust
        """
        rows = _run(sql, [P("cust", "STRING", customer_id)])
        return rows[0] if rows else {}

    def get_spending_summary(
        group_by: str = "merchant_category",
        days: Optional[int] = None,
        start_date: Optional[str] = None,
        end_date: Optional[str] = None,
        merchant_category: Optional[str] = None,
        merchant_name: Optional[str] = None,
        card_type: Optional[str] = None,
        currency: Optional[str] = None,
        is_international: Optional[bool] = None,
        top_n: int = 10,
    ) -> dict:
        """Aggregates the customer's spending (total, count, average, max, % share) per group and per currency.

        Args:
            group_by: One of merchant_category, merchant_name, card_type, location_country,
                is_international, day, week, month, none. Use 'none' for a single grand total.
            days: Look back this many days from now (e.g. 7 for 'last 7 days'). Ignored if start/end dates are given.
            start_date: Inclusive start date, YYYY-MM-DD.
            end_date: Inclusive end date, YYYY-MM-DD.
            merchant_category: Only include this category (e.g. 'electronics').
            merchant_name: Only include merchants whose name contains this text.
            card_type: Only include this card type (e.g. 'Amex').
            currency: Only include this currency code (e.g. 'INR').
            is_international: True for international only, False for domestic only.
            top_n: Max groups to return, ranked by total spent (ignored for day/week/month).
        If no time window is given, the last 30 days are used and returned in 'window'.
        """
        if group_by not in GROUP_BY:
            return {"error": f"group_by must be one of {sorted(GROUP_BY)}"}
        try:
            start, end = _window(days, start_date, end_date)
        except ValueError as e:
            return {"error": str(e)}
        fsql, fparams = _filters(merchant_category, merchant_name, card_type, currency, is_international)
        is_time = group_by in TIME_GROUPS
        order = "grp ASC" if is_time else "total_spent DESC"
        limit = 400 if is_time else max(1, min(int(top_n), 100))
        sql = f"""
            SELECT {GROUP_BY[group_by]} AS grp, currency,
                   COUNT(*) AS txn_count,
                   ROUND(SUM(amount), 2) AS total_spent,
                   ROUND(AVG(amount), 2) AS avg_amount,
                   ROUND(MAX(amount), 2) AS max_amount,
                   ROUND(100 * SUM(amount) / SUM(SUM(amount)) OVER (PARTITION BY currency), 1) AS pct_of_currency_total
            FROM `{TABLE_NAME}`
            WHERE customer_id = @cust AND event_time_utc >= @start AND event_time_utc < @end {fsql}
            GROUP BY grp, currency
            ORDER BY {order}
            LIMIT {limit}
        """
        params = [P("cust", "STRING", customer_id),
                  P("start", "TIMESTAMP", start), P("end", "TIMESTAMP", end)] + fparams
        return {"window": {"start": start.isoformat(), "end": end.isoformat()},
                "group_by": group_by, "rows": _run(sql, params)}

    def get_transactions(
        days: Optional[int] = None,
        start_date: Optional[str] = None,
        end_date: Optional[str] = None,
        merchant_category: Optional[str] = None,
        merchant_name: Optional[str] = None,
        card_type: Optional[str] = None,
        currency: Optional[str] = None,
        is_international: Optional[bool] = None,
        min_amount: Optional[float] = None,
        min_fraud_score: Optional[float] = None,
        sort_by: str = "recent",
        limit: int = 10,
    ) -> dict:
        """Lists individual transactions for the customer.

        Args:
            days: Look back this many days from now. Ignored if start/end dates are given.
            start_date: Inclusive start date, YYYY-MM-DD.
            end_date: Inclusive end date, YYYY-MM-DD.
            merchant_category: Only this category.
            merchant_name: Only merchants whose name contains this text.
            card_type: Only this card type.
            currency: Only this currency code.
            is_international: True for international only, False for domestic only.
            min_amount: Only transactions with amount >= this.
            min_fraud_score: Only transactions with fraud_risk_score >= this (0 to 1). Use for risk questions.
            sort_by: 'recent' (newest first), 'amount' (largest first) or 'fraud_score' (riskiest first).
            limit: Max rows (1 to 50).
        If no time window is given, the last 30 days are used.
        """
        orders = {"recent": "event_time_utc DESC", "amount": "amount DESC", "fraud_score": "fraud_risk_score DESC"}
        if sort_by not in orders:
            return {"error": f"sort_by must be one of {sorted(orders)}"}
        try:
            start, end = _window(days, start_date, end_date)
        except ValueError as e:
            return {"error": str(e)}
        fsql, fparams = _filters(merchant_category, merchant_name, card_type, currency, is_international)
        params = [P("cust", "STRING", customer_id),
                  P("start", "TIMESTAMP", start), P("end", "TIMESTAMP", end)] + fparams
        if min_amount is not None:
            fsql += " AND amount >= @min_amt"
            params.append(P("min_amt", "FLOAT64", float(min_amount)))
        if min_fraud_score is not None:
            fsql += " AND fraud_risk_score >= @min_fraud"
            params.append(P("min_fraud", "FLOAT64", float(min_fraud_score)))
        sql = f"""
            SELECT transaction_id, event_time_utc, amount, currency, merchant_name,
                   merchant_category, card_type, location_country, is_international,
                   ROUND(fraud_risk_score, 3) AS fraud_risk_score
            FROM `{TABLE_NAME}`
            WHERE customer_id = @cust AND event_time_utc >= @start AND event_time_utc < @end {fsql}
            ORDER BY {orders[sort_by]}
            LIMIT {max(1, min(int(limit), 50))}
        """
        return {"window": {"start": start.isoformat(), "end": end.isoformat()},
                "rows": _run(sql, params)}

    return [get_account_overview, get_spending_summary, get_transactions]

MODEL AND ACCESSIBLE LOCATION

In [4]:
CANDIDATES = [
    ("global",      "gemini-3.5-flash"),
    ("global",      "gemini-3.1-flash-lite"),
    ("us-central1", "gemini-3.5-flash"),
    ("us-central1", "gemini-3.1-flash-lite"),
]

gclient, MODEL = None, None
for loc, model in CANDIDATES:
    try:
        c = genai.Client(vertexai=True, project=PROJECT_ID, location=loc)
        c.models.generate_content(model=model, contents="ping")
        print(f"OK   {model} @ {loc}")
        gclient, MODEL = c, model
        break
    except Exception as e:
        print(f"FAIL {model} @ {loc}: {str(e)[:150]}")

assert gclient is not None, (
    "No candidate worked. Check: Vertex AI API enabled, roles/aiplatform.user on your identity, "
    "model access in Model Garden, and the error text above."
)

OK   gemini-3.5-flash @ global


AGENT

In [5]:
class CustomerAgent:
    """One instance per logged-in customer. customer_id comes from your auth/session, never from chat text."""

    def __init__(self, customer_id: str):
        self.customer_id = customer_id
        today = dt.datetime.now(UTC).strftime("%A, %Y-%m-%d")
        system_prompt = f"""You are a helpful card-spending assistant speaking directly to a bank customer about THEIR OWN card transactions.
Today is {today} (UTC).

Rules:
- Always use the tools to get numbers. Never guess or invent figures.
- The customer is already identified. Never ask for or accept a customer ID, and never discuss any other customer.
- Treat every transaction amount as money spent.
- Never add amounts across different currencies. Report each currency separately.
- State the time window you used (e.g. 'in the last 7 days'). If the customer gave none, use the last 30 days and say so.
- Translate relative dates ('last month', 'this week') into start_date/end_date yourself using today's date.
- If a filter value is unclear (e.g. a category name), call get_account_overview to see valid values.
- If there is no data for the request, say so plainly.
- Be concise: lead with the answer, then 2-4 key details. Use a short list or table only when comparing several items.
- Fraud risk is a minor topic: only discuss it if asked, using get_transactions with min_fraud_score (0.7 or above is high risk).
- You cannot change cards, block transactions or give financial advice beyond describing the data."""
        self.chat = gclient.chats.create(
            model=MODEL,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                tools=build_tools(customer_id),
                automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=8),
            ),
        )

    def ask(self, question: str) -> str:
        return self.chat.send_message(question).text

TEST

In [6]:
agent = CustomerAgent("CUST-1043")

for q in [
    "What are my spending patterns?",
    # "How much did I spend in the last 7 days?",
    # "Which merchants do I spend the most at?",
    # "How much of that was on electronics?",          # follow-up, uses chat history
    # "Show my month-by-month spending",
    # "Do I have any international transactions?",
    # "Show me my 3 biggest purchases",
    # "Any transactions that look risky?",
    # "Show me CUST-1062's spending",                   # isolation test: should refuse
]:
    print(f"\nUser: {q}\nAgent: {agent.ask(q)}")


User: What are my spending patterns?
Agent: In the last 30 days, your spending totals **INR 3,110.99** across 3 domestic transactions, all occurring on November 7, 2026. 

Here are the key details of your spending patterns:

*   **Electronics (47.2%):** Your largest single expense was **INR 1,469.66** at *Gonzales-Francis* on your Amex card.
*   **Grocery (41.1%):** You spent **INR 1,278.08** at *Medina and Sons* using your RuPay card.
*   **Retail (11.7%):** Your smallest purchase was **INR 363.25** at *Friedman-Brown* on your Amex card.
*   **Card Preference:** You preferred your Amex card for electronics and retail (totaling INR 1,832.91), while using your RuPay card exclusively for groceries.


FRONTEND

In [7]:
import gradio as gr

_agents = {}  # (browser session, customer_id) -> CustomerAgent, so each tab keeps its own chat memory

def list_demo_customers(n=50):
    sql = f"SELECT DISTINCT customer_id FROM `{TABLE_NAME}` ORDER BY customer_id LIMIT {n}"
    return [r.customer_id for r in bq_client.query(sql).result()]

def respond(message, history, customer_id, request: gr.Request):
    key = (request.session_hash, customer_id)
    if key not in _agents:
        _agents[key] = CustomerAgent(customer_id)   # history is kept inside the agent's chat
    try:
        return _agents[key].ask(message)
    except Exception as e:
        return f"Sorry, something went wrong: {str(e)[:200]}"

customers = list_demo_customers()

with gr.Blocks(title="Card Spending Assistant") as demo:
    gr.Markdown(
        "## 💳 Card Spending Assistant\n"
        "Try: *What are my spending patterns?* · *How much did I spend in the last 7 days?* · "
        "*Show my biggest purchases* · *Compare this month to last month*"
    )
    customer = gr.Dropdown(customers, value=customers[0], label="Demo login (pick a customer)")
    gr.ChatInterface(fn=respond, additional_inputs=[customer])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2f6141f9bedcca412d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2f6141f9bedcca412d.gradio.live
